# TalkTalk NBA — Top-50 most-impacted customers

Loads the artefact produced by `train.ipynb`, scores every customer, ranks by churn probability, attaches the strongest reason codes (SHAP if available, otherwise a fast feature-importance × normalised feature-value fallback), recommends an NBA, and writes:

- `top_50_customers.json` — upload via *Lovable → Model → Import results*

All input files (`customer_info`, `calls`, `usage`) sit next to this notebook (`DATA = '.'`).

## 1 · Setup

```bash
pip install pandas numpy scikit-learn pyarrow fastparquet xgboost
# optional, for SHAP-quality reason codes:
pip install shap
```

In [1]:
from __future__ import annotations
import json, pickle
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('shap not installed — using feature-importance fallback for reasons')

DATA = Path('../data')
OUT  = Path('../out')
ID   = 'unique_customer_identifier'

ART = OUT / 'model_artefact.pkl'
if not ART.exists():
    raise SystemExit(f'Missing {ART} — run train.ipynb first')

with open(ART, 'rb') as f:
    bundle = pickle.load(f)
model      = bundle['model']
features   = bundle['features']
threshold  = bundle['threshold']
model_type = bundle['model_type']
print(f'Loaded {model_type} with {len(features)} features (threshold={threshold:.2f})')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded XGBoost with 15 features (threshold=0.41)


## 2 · Rebuild the same features used in training

In [2]:
def load(name: str) -> pd.DataFrame:
    pq, csv = DATA / f'{name}.parquet', DATA / f'{name}.csv'
    if pq.exists():  return pd.read_parquet(pq)
    if csv.exists(): return pd.read_csv(csv)
    raise FileNotFoundError(name)

customer_info = load('customer_info')
calls         = load('calls')
usage         = load('usage')

calls['event_date'] = pd.to_datetime(calls['event_date'], errors='coerce')
max_call = calls['event_date'].max()
recent   = calls[calls['event_date'] >= (max_call - pd.Timedelta(days=90))]

# Coerce numeric-looking columns that may have been read as strings
for col in ('hold_time_seconds', 'talk_time_seconds'):
    if col in calls.columns:
        calls[col] = pd.to_numeric(calls[col], errors='coerce')
for col in ('usage_download_mbs', 'usage_upload_mbs'):
    if col in usage.columns:
        usage[col] = pd.to_numeric(usage[col], errors='coerce')

loyalty_90d  = recent.groupby(ID).size().rename('loyalty_calls_90d')
avg_hold     = calls.groupby(ID)['hold_time_seconds'].mean().rename('avg_hold_seconds')
avg_talk     = calls.groupby(ID)['talk_time_seconds'].mean().rename('avg_talk_seconds')
avg_download = usage.groupby(ID)['usage_download_mbs'].mean().rename('avg_download_mbs')
avg_upload   = usage.groupby(ID)['usage_upload_mbs'].mean().rename('avg_upload_mbs')

df = (customer_info
      .merge(loyalty_90d,  on=ID, how='left')
      .merge(avg_hold,     on=ID, how='left')
      .merge(avg_talk,     on=ID, how='left')
      .merge(avg_download, on=ID, how='left')
      .merge(avg_upload,   on=ID, how='left'))

X = df.reindex(columns=features).copy()
for c in X.columns:
    if X[c].dtype == object:
        X[c] = X[c].astype('category').cat.codes
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
print(f'Scoring {len(X):,} customers…')

Scoring 3,545,538 customers…


## 3 · Score everyone

In [3]:
probs = model.predict_proba(X)[:, 1]
df['churn_prob'] = probs
df['churn_prob'].describe()

count    3.545538e+06
mean     4.753498e-01
std      3.088702e-01
min      2.238861e-03
25%      1.673660e-01
50%      4.808829e-01
75%      7.593885e-01
max      9.999546e-01
Name: churn_prob, dtype: float64

## 4 · Reason codes

If SHAP is available we compute true Shapley values for the top 50; otherwise we approximate with `feature_importance × min-max-normalised feature value`.

In [4]:
def reason_codes_fallback(row_x: pd.Series) -> list:
    fi = getattr(model, 'feature_importances_', None)
    if fi is None: return []
    rng = (X.max() - X.min()).replace(0, 1)
    norm = (row_x - X.min()) / rng
    contribs = [{'feature': f, 'impact': float(w * n)} for f, w, n in zip(features, fi, norm)]
    contribs.sort(key=lambda d: d['impact'], reverse=True)
    return contribs[:3]

reasons_by_idx = {}
top_idx = df['churn_prob'].nlargest(50).index

if HAS_SHAP:
    print('Computing SHAP values for top 50…')
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X.loc[top_idx])
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    for i, idx in enumerate(top_idx):
        ranked = sorted(zip(features, shap_vals[i]), key=lambda t: abs(t[1]), reverse=True)[:3]
        reasons_by_idx[idx] = [{'feature': f, 'impact': float(v)} for f, v in ranked]
print(f'Reason codes ready for {len(reasons_by_idx) or 50} customers')

Computing SHAP values for top 50…
Reason codes ready for 50 customers


## 5 · NBA rule book

Tiny mapping from the dominant reason to the recommended next-best-action. Edit freely — these are mirrored in Lovable's NBA rules page.

In [5]:
NBA_RULES = {
    'loyalty_calls_90d':   'Retention call · 15% off 12 mo',
    'avg_hold_seconds':    'Priority care queue + £20 credit',
    'avg_talk_seconds':    'Care follow-up + service review',
    'ooc_days':            'Loyalty re-contract · 24 mo £5/mo off',
    'tenure_days':         'Tenure reward · service credit',
    'speed':               'Free fibre upgrade audit',
    'line_speed':          'Free line speed audit',
    'avg_download_mbs':    'Unlimited data add-on',
    'avg_upload_mbs':      'Upload boost add-on',
    'contract_dd_cancels': 'Billing welfare check + payment plan',
    'dd_cancel_60_day':    'Urgent save call · payment hold',
    'crm_package_name':    'Right-size package recommendation',
    'technology':          'Tech migration offer (FTTC → FTTP)',
    'sales_channel':       'Channel-aware retention offer',
    'contract_status':     'Standard retention offer',
}

## 6 · Build the top-50 records

Categorical features (e.g. `technology = 'FTTP'`, `crm_package_name`) are kept as **strings** in the exported `features` payload — they are only encoded to integer codes for the model itself. `expected_save_gbp` is a placeholder estimate (`prob × £25 ARPU × 12 months × 50% success`); the schema does not carry ARPU so we use a flat assumption.

In [6]:
top = df.nlargest(50, 'churn_prob')[[ID, 'churn_prob'] + features].reset_index()
ASSUMED_ARPU = 25.0
SUCCESS_RATE = 0.5

def _jsonable(v):
    """Return a JSON-safe value: numbers stay numeric, strings (e.g. 'FTTP') stay strings, NaN→None."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    return str(v)

# Use the original (untransformed) customer_info row so categorical features
# like technology='FTTP' come through as strings rather than category codes.
raw_lookup = customer_info.drop_duplicates(ID).set_index(ID)

records = []
for i, row in top.iterrows():
    orig_idx = row['index']
    rs = reasons_by_idx.get(orig_idx) or reason_codes_fallback(X.loc[orig_idx])
    dominant = rs[0]['feature'] if rs else 'tenure_days'
    nba = NBA_RULES.get(dominant, 'Standard retention offer')
    expected_save = float(row['churn_prob']) * ASSUMED_ARPU * 12 * SUCCESS_RATE
    cust_id = str(row[ID])
    raw_row = raw_lookup.loc[cust_id] if cust_id in raw_lookup.index else None
    feature_payload = {}
    for f in features:
        if raw_row is not None and f in raw_row.index:
            val = raw_row[f]
        else:
            val = row[f]
        feature_payload[f] = _jsonable(val)
    records.append({
        'customer_id':       cust_id,
        'rank':              int(i + 1),
        'churn_prob':        float(row['churn_prob']),
        'reason_codes':      rs,
        'recommended_nba':   nba,
        'expected_save_gbp': round(expected_save, 2),
        'features':          feature_payload,
    })

records[:3]

[{'customer_id': '5ce86ea491705d4a7cbd0e676988d7ffbd1283247aa1580f56d4f676703d0111',
  'rank': 1,
  'churn_prob': 0.9999545812606812,
  'reason_codes': [{'feature': 'speed', 'impact': 4.098898410797119},
   {'feature': 'dd_cancel_60_day', 'impact': 1.3899297714233398},
   {'feature': 'contract_dd_cancels', 'impact': 1.3433643579483032}],
  'recommended_nba': 'Free fibre upgrade audit',
  'expected_save_gbp': 149.99,
  'features': {'ooc_days': 144.0,
   'tenure_days': 3562.0,
   'speed': 1000.0,
   'line_speed': 0.0,
   'contract_dd_cancels': 1.0,
   'dd_cancel_60_day': 1.0,
   'loyalty_calls_90d': None,
   'avg_hold_seconds': 1480.0,
   'avg_talk_seconds': 237.0,
   'avg_download_mbs': None,
   'avg_upload_mbs': None,
   'technology': 'FTTP',
   'sales_channel': 'Retail',
   'crm_package_name': 'Ultra Fibre Optic',
   'contract_status': '06 OOC'}},
 {'customer_id': 'be400ed8e14560b924af5221607cf9e6353429321c06107ab51766a461cfd4d6',
  'rank': 2,
  'churn_prob': 0.9999500513076782,
  're

## 7 · Write `top_50_customers.json`

In [7]:
payload = {
    'model_type': model_type,
    'threshold':  threshold,
    'scored_n':   int(len(df)),
    'customers':  records,
}
with open(OUT / 'top_50_customers.json', 'w') as f:
    json.dump(payload, f, indent=2, default=str)

print(f'✓ Scored {len(df):,} customers, picked top {len(records)}.')
print('✓ top_50_customers.json written — upload via Lovable → Model → Import results.')

✓ Scored 3,545,538 customers, picked top 50.
✓ top_50_customers.json written — upload via Lovable → Model → Import results.
